# Déploiement fidèle sur SpiNNaker

Ce notebook lance `spinnaker_faithful_eval.py` dans le noyau Jupyter, là où `pyNN.spiNNaker` est installé.
Exécutez les cellules dans l'ordre. Résultats dans `results_spinnaker/`.

## 1. Dossier de travail et dépendances

In [ ]:
import importlib
import os
import site
import subprocess
import sys

if not os.path.exists("spinnaker_plan.json") and os.path.exists("../spinnaker_plan.json"):
    os.chdir("..")
assert os.path.exists("spinnaker_plan.json"), "spinnaker_plan.json introuvable : extraire sands_bundle.tar.gz dans work/"
print("Dossier de travail :", os.getcwd())


def pip_install(*packages):
    """Installe dans l'environnement du noyau (et non dans celui du terminal)."""
    command = [sys.executable, "-m", "pip", "install", "--quiet", *packages]
    if subprocess.call(command) != 0:
        subprocess.check_call(command[:4] + ["--user"] + command[4:])
    importlib.invalidate_caches()
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


try:
    import torch
except ImportError:
    pip_install("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
try:
    import spikingjelly
except ImportError:
    pip_install("--no-deps", "spikingjelly")

## 2. Vérification de l'environnement

In [ ]:
import pyNN.spiNNaker as sim
import spikingjelly
import torch

sys.path.insert(0, os.path.join(os.getcwd(), "SpiNNaker"))
import spinnaker_energy
import spinnaker_faithful_eval as faithful

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("pyNN        :", sim.__name__, "OK")
print("neurone     :", faithful.delta_neuron_model(sim).__name__)
print("script      :", "à jour (v5.1)" if hasattr(faithful, "count_cores") else "ANCIENNE VERSION, à remplacer")

import configparser

config = configparser.ConfigParser(strict=False)
config.read([os.path.expanduser("~/.spynnaker.cfg"), "spynnaker.cfg"])
print("énergie     :", config.get("Reports", "write_energy_report", fallback="False"), "(doit valoir True)")

## 3. Expérience de diagnostic (déjà faite le 12/11)

A (128 neurones/cœur, facteur 10) : 812 705 spikes tardifs, carte 13 %. B (facteur 50) : 0 tardif, carte = virgule fixe. C (32 neurones/cœur, facteur 10) : 116 717 tardifs, accord 87 %. Inutile de relancer.

In [ ]:
import json

plan = json.load(open("spinnaker_plan.json"))
cible = [e for e in plan if e["dataset"] == "shd" and e["architecture"] == "SpikingMLP"
         and int(e["T"]) == 32 and float(e["Vth"]) == 1.0]
json.dump(cible, open("plan_diagnostic.json", "w"))

base = ["--plan", "plan_diagnostic.json"]
faithful.main(base + ["--results-dir", "diag4_A", "--neurons-per-core", "128"])
faithful.main(base + ["--results-dir", "diag4_B", "--neurons-per-core", "128", "--time-scale-factor", "50"])
faithful.main(base + ["--results-dir", "diag4_C"])

## 4. Vérification sans la carte (quelques minutes)

Pour chaque run : bits d'entrée, facteur de ralentissement minimal (pour que les cœurs suivent **et** que le réseau tienne sur la carte), temps estimé du cœur le plus chargé, cœurs demandés. La cellule en déduit le facteur commun `FACTEUR` et le run le plus chargé, utilisés aux sections 5 et 6.

In [ ]:
for dataset in ["smnist", "shd", "ssc", "nmnist"]:
    faithful.main(["--dataset", dataset, "--emulate", "--time-scale-factor", "auto"])

import csv
with open("results_spinnaker_v5/summary_emulation.csv", newline="", encoding="utf-8") as f:
    emulation = {r["run_id"]: r for r in csv.DictReader(f)}
ok = {k: r for k, r in emulation.items() if r["status"] == "EMULATED"}
for r in ok.values():
    print(f"{r['run_id']:<38} bits={r['input_bits']:>2} facteur={r['needed_time_scale_factor']:>4} "
          f"coût={r['core_cost_ms']:>7} ms spikes/cœur={r['peak_core_spikes']:>6} cœurs={r['n_cores']:>5}")
for k, r in emulation.items():
    if r["status"] != "EMULATED":
        print(f"{k:<38} {r['status']} : {r['error_msg']}")

FACTEUR = str(max(int(r["needed_time_scale_factor"]) for r in ok.values()))
PIRE = max(ok.values(), key=lambda r: (int(r["needed_time_scale_factor"]), float(r["core_cost_ms"])))
print(f"\n{len(ok)}/{len(emulation)} runs déployables | facteur commun : {FACTEUR} | "
      f"run le plus chargé : {PIRE['run_id']} ({PIRE['core_cost_ms']} ms au facteur {PIRE['needed_time_scale_factor']})")

## 5. Test du run le plus chargé (quelques minutes)

Attendu : `fiable=True`, `tardifs=0`, `accord` ≥ 90 %. Si c'est le cas, lancer la section 6. Sinon, ne pas lancer la section 6 et m'envoyer la ligne avec la colonne `late_by_population` affichée en dessous.

In [ ]:
import json

plan = json.load(open("spinnaker_plan.json"))
cible = [e for e in plan if faithful.tee.run_id_of(e) == PIRE["run_id"]]
json.dump(cible, open("plan_test_pire.json", "w"))
faithful.main(["--plan", "plan_test_pire.json", "--results-dir", f"test_v5_facteur{FACTEUR}",
               "--time-scale-factor", FACTEUR])

with open(f"test_v5_facteur{FACTEUR}/summary_benchmark.csv", newline="", encoding="utf-8") as f:
    test = list(csv.DictReader(f))[-1]
print("tardifs par population :", test["late_by_population"] or "aucun")

## 6. Tous les modèles sur la carte

Même facteur pour tous les runs (`FACTEUR`, calculé en section 4), pour que les énergies restent comparables. Durée : environ 3 h + 0,2 h × FACTEUR / 10. La cellule peut être relancée : les runs déjà réussis sont sautés.

In [ ]:
print("Facteur utilisé :", FACTEUR)
for dataset in ["smnist", "shd", "ssc", "nmnist"]:
    faithful.main(["--dataset", dataset, "--time-scale-factor", FACTEUR])

## 7. Précision et fiabilité

Un run est exploitable si `fiable` vaut True et `accord` est d'au moins 90 %.

In [ ]:
import csv

with open("results_spinnaker_v5/summary_benchmark.csv", newline="", encoding="utf-8") as f:
    rows = {r["run_id"]: r for r in csv.DictReader(f)}

print(f"{'run':<38} {'pytorch':>7} {'v.fixe':>7} {'carte':>6} {'accord':>6} {'n/cœur':>6} {'perdus':>7} {'tardifs':>7} {'retards':>7} {'fiable':>6}")
a_refaire = []
for row in rows.values():
    print(
        f"{row['run_id']:<38} {row['accuracy_pytorch_pct']:>7} {row['accuracy_fixed_point_pct']:>7} "
        f"{row['accuracy_pct']:>6} {row['agreement_fixed_point_pct']:>6} {row['min_neurons_per_core']:>6} "
        f"{row['lost_input_packets']:>7} {row['late_spikes']:>7} {row['timer_overruns']:>7} {row['reliable']:>6}"
    )
    if row["status"] != "SUCCESS" or row["reliable"] != "True":
        a_refaire.append(row["run_id"])
print(f"\n{len(rows) - len(a_refaire)}/{len(rows)} runs fiables. À refaire : {a_refaire}")
print("Source de la provenance :", {r["provenance_source"][:40] for r in rows.values()})

## 8. Énergie réelle et tableau final

Écrit `results_spinnaker_v5/spinnaker_energy.csv` et, si `theoretical_energy_results.csv` est présent, `final_energy_results.csv`.

In [ ]:
spinnaker_energy.main(["--results-dir", "results_spinnaker_v5"])